# Atari Space Invaders with RL-Baselines-3Zoo

**Deep Q-learning** replaces Q-table with a neural network to hadle large environments that involve continous actions and states. The Q-value function is parametrized, and for a given state the Q-values for each possible action are approximated by a neural network - **Deep Q-Network**. The epsilon-greedy policy is still use to select the action to be taken.

**Pre-processing the input** by reducing the state space, cropping a part of the screen and switching to grayscale. Handle the problem of **temporal limitation** by stacking four frames together, processig them by three convolutional layers, which allows the capture and processing of spatial relationships in images, and explore temporal properties across those frames

If combine non-le=inear Q-value functiona nd bootstraping when updating the targets with existing extimates in the same Neural Network leads to unstabiity, to stabilize the training implement:

1. Experience replay in a replay buffer, which stores experience tuples - state, action, reward and next state, to sample a small batch of tuples, which allows the agent to learn from the same experience multiple times and avoid catastrophic forgetting

2. Fixed Q-Target for calculating the loss = Q-Target - current estimation of Q-value

3. Double Deep Q-Learning, use a separate network with fixed parameters for estimating the TD Target, copy and update the target network parameters and
* use the DQN network to select the best action to take for the next state
* use the Target network to calculate the target Q-value of taking the action at the next state

as well as "new kids on the block" extentions like  Prioritized Experience Replay, Dueling Deep Q-Learning etc.


## Package installation

RL Baselines3 Zoo is a training framework based on Stable-Baselines3 that provides scripts for training, evaluating, tuning and plotting, a vanilla version of Deep Q-Learing with no extensions like Double-DQN, Dueling-DQN or Prioritized Experience Replay

In [ ]:
# install updated RL-Baselines3-zoo
!pip install git+https://github.com/DLR-RM/rl-baselines3-zoo

In [ ]:
# install dependencies for reinforcement learning and game environments such as Atari Space Invaders
!apt-get install swig cmake ffmpeg

In [ ]:
# install atari packages to be able to use Atari games in Gymnasium, and accept-rom-license
!pip install gymnasium[atari]
!pip install gymnasium[accept-rom-license]

In [ ]:
!apt-get update
!apt-get install -y ffmpeg xvfb
!pip install pyvirtualdisplay
!pip install pyglet==1.5.27
!pip install PyOpenGL PyOpenGL_accelerate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd drive/MyDrive/ColabNotebooks/DeepRL/deep_q_learning

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/ColabNotebooks/DeepRL/deep_q_learning


In [ ]:
from pyvirtualdisplay import Display

# start a virtual display that creates an off-screen display Xvfb that lets Gym render frames
virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

## Training

Hyperparameter config file, dqn.yaml, that constain training hyperparameters.

* Use Atari Wrapper to preprocess the inpput: stack 4 grayscale frames
* Use CNN Policy, a Convolutional layer to process the frames
* Memory = Experience Replay, buffer_size size is the amount of experience steps  saved to train the agent again

In [ ]:
!ls

atari_invaderts.ipynb  dqn_0.yml  rlzoo_optimze_optuna_results
best_params.yml        dqn.yml	  rlzoo_optuna_optimized


In [ ]:
# start training with algorithm dqn, with hyperparameter config dqn.yml and save the model in logs folder
!python -m rl_zoo3.train \
--algo dqn \
--env SpaceInvadersNoFrameskip-v4 \
-f logs/ \
-c dqn.yml


Streaming output truncated to the last 5000 lines.
|    loss             | 0.0641   |
|    n_updates        | 830053   |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 3.99e+03 |
|    ep_rew_mean      | 682      |
|    exploration_rate | 0.01     |
| time/               |          |
|    episodes         | 9124     |
|    fps              | 253      |
|    time_elapsed     | 13521    |
|    total_timesteps  | 3421917  |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0269   |
|    n_updates        | 830479   |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 3.98e+03 |
|    ep_rew_mean      | 684      |
|    exploration_rate | 0.01     |
| time/               |          |
|    episodes         | 9128     |
|    fps              | 253      |
|    time_elapsed     | 13530    |
|   

## Evaluation

In [ ]:
# model saved in best_model.zip, rename to SpaceInvadersNoFrameskip-v4 for the upload to hub to work
!ls logs/dqn/SpaceInvadersNoFrameskip-v4_1/

0.monitor.csv	evaluations.npz		     SpaceInvadersNoFrameskip-v4.zip
best_model.zip	SpaceInvadersNoFrameskip-v4


In [ ]:
# record video while evaluating the train agent for specified number of steps
!python -m rl_zoo3.enjoy \
  --algo dqn \
  --env SpaceInvadersNoFrameskip-v4 \
  --folder logs/ \
  --n-timesteps 5000 \
  --no-render \
  --load-best


2025-10-22 13:47:22.387954: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761140842.422021    7289 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761140842.432050    7289 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1761140842.456247    7289 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1761140842.456286    7289 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1761140842.456290    7289 computation_placer.cc:177] computation placer alr

In [ ]:
# continue training from flag to load the pretrained model, specifing the full path
# to the checkpoint to resume from with new additional -n steps
!python -m rl_zoo3.train \
  --algo dqn \
  --env SpaceInvadersNoFrameskip-v4 \
  -f logs/ \
  -c dqn.yml \
  -i logs/dqn/SpaceInvadersNoFrameskip-v4_1/best_model.zip \
  -n 500000

Streaming output truncated to the last 5000 lines.
|    learning_rate    | 0.0001   |
|    loss             | 0.00391  |
|    n_updates        | 29762    |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 2.54e+03 |
|    ep_rew_mean      | 241      |
|    exploration_rate | 0.01     |
| time/               |          |
|    episodes         | 1132     |
|    fps              | 305      |
|    time_elapsed     | 720      |
|    total_timesteps  | 220109   |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.00356  |
|    n_updates        | 30027    |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 2.55e+03 |
|    ep_rew_mean      | 243      |
|    exploration_rate | 0.01     |
| time/               |          |
|    episodes         | 1136     |
|    fps              | 305      |
|   

In [ ]:
!python -m rl_zoo3.enjoy \
  --algo dqn \
  --env SpaceInvadersNoFrameskip-v4 \
  --folder logs/ \
  --n-timesteps 5000 \
  --no-render \
  --exp-id 2 \
  --load-best

2025-10-22 15:04:26.859581: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761145466.879639   26751 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761145466.885777   26751 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1761145466.901428   26751 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1761145466.901462   26751 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1761145466.901466   26751 computation_placer.cc:177] computation placer alr

In [ ]:
!pip install --upgrade gymnasium[atari] ale-py autorom


In [ ]:
!pip uninstall -y gymnasium gymnasium[atari] ale-py autorom
!pip install "gymnasium[atari]" ale-py autorom
!python -m autorom.accept-rom-license --accept-license

In [ ]:
!pip install --upgrade "gymnasium[atari, accept-rom-license]"
!pip install stable-baselines3[extra]
!pip install rl-zoo3 pyvirtualdisplay

In [ ]:
!pip install ale-py

In [ ]:
import gymnasium as gym
import ale_py

env_id = "ALE/SpaceInvaders-v5"
env = gym.make(env_id, render_mode="rgb_array")

print(env.observation_space)

Box(0, 255, (210, 160, 3), uint8)


In [ ]:
env_id = "SpaceInvaders-v4"
env = gym.make(env_id, render_mode="rgb_array")

print(env.observation_space)

Box(0, 255, (210, 160, 3), uint8)


Problems with training and evaluating/recoding environments are solved with wrappers:observation shape (84, 84, 1) image has been resized to 84x84 and converted to grayscale, need to add a VecFrameStack wrapper after the AtariWrapper in order to stack 4 consecutive frames, (4, 84, 84) observation provides the agent with information about motion.

The standard RecordVideo wrapper VecVideoRecorder is designed for single, non-vectorized environments. For a VecEnv, the stable-baselines3 library provides VecVideoRecorder, which handles recording from vectorized environments.
DummyVecEnv is used to create a "vectorized" environment from a single environment, which is necessary for the VecFrameStack to function.
VecFrameStack(env, n_stack=4) takes the environment, which is now outputting (84, 84, 1) observations, and stacks four consecutive frames together to produce the (4, 84, 84) shape required by the trained model

Need ale-py package for modern gymnasium Atari environment "ALE/SpaceInvaders-v5" from Arcade Learning Environment used by rl_zoo3

In [ ]:
from gymnasium.wrappers import RecordVideo
from stable_baselines3 import DQN
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecVideoRecorder
import gymnasium as gym
import ale_py
import os

# Import necessary wrappers
from stable_baselines3.common.atari_wrappers import AtariWrapper

# Start virtual display
virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()


# path to your best model
best_model_path = 'logs/dqn/SpaceInvadersNoFrameskip-v4_2/best_model.zip'
model = DQN.load(best_model_path)

env_id = "ALE/SpaceInvaders-v5"

# Create the environment with a RecordVideo wrapper and Atari wrappers
video_folder = "./videos/"
os.makedirs(video_folder, exist_ok=True)


# VecFrameStack requires a VecEnv, so we use DummyVecEnv to wrap the single env.
env = DummyVecEnv([lambda: AtariWrapper(gym.make(env_id, render_mode="rgb_array"))])
env = VecFrameStack(env, n_stack=4)
env = VecVideoRecorder(env, video_folder,
                       record_video_trigger=lambda step: step == 0,
                       video_length=5000,
                       name_prefix=env_id.replace('/', '-'))

# when the model is wraped env.rest() only returns observaition and no info
obs = env.reset()
for _ in range(5000):
    action, _states = model.predict(obs, deterministic=True)
    obs, rewards, done, info = env.step(action)
    if done:
        obs = env.reset()
env.close()

print(f"Video saved to: {video_folder}")

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/vec_env/vec_video_recorder.py:155: UserWarning: WARN: Unable to save last video! Did you call close()?
  logger.warn("Unable to save last video! Did you call close()?")


Saving video to /content/drive/MyDrive/ColabNotebooks/DeepRL/deep_q_learning/videos/ALE-SpaceInvaders-v5-step-0-to-step-5000.mp4
Moviepy - Building video /content/drive/MyDrive/ColabNotebooks/DeepRL/deep_q_learning/videos/ALE-SpaceInvaders-v5-step-0-to-step-5000.mp4.
Moviepy - Writing video /content/drive/MyDrive/ColabNotebooks/DeepRL/deep_q_learning/videos/ALE-SpaceInvaders-v5-step-0-to-step-5000.mp4



Moviepy - Done !
Moviepy - video ready /content/drive/MyDrive/ColabNotebooks/DeepRL/deep_q_learning/videos/ALE-SpaceInvaders-v5-step-0-to-step-5000.mp4
Video saved to: ./videos/


In [ ]:
# display the recorded video
import io
import os
import base64
from IPython.display import HTML

def show_video(video_path):
    mp4 = open(video_path,'rb').read()
    data_url = "data:video/mp4;base64," + base64.b64encode(mp4).decode()
    return HTML(f'<video width=480 controls><source src="{data_url}" type="video/mp4"></video>')

# Example
show_video("videos/ALE-SpaceInvaders-v5-step-0-to-step-5000.mp4")


## Publish

rl_zoo3.push_to_hub evaluates, record a replay, generate a model card of the agent and push it to the hub

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
!git config --global credential.helper store

In [ ]:
# push to hub
!python -m rl_zoo3.push_to_hub \
--algo dqn  \
--env SpaceInvadersNoFrameskip-v4 \
--repo-name dqn-SpaceInvadersNoFrameskip-v4 \
-orga elliemci \
--exp-id 2 \
-f logs \
--load-best

2025-10-22 17:04:30.662065: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761152670.683099   58221 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761152670.692803   58221 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1761152670.715594   58221 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1761152670.715623   58221 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1761152670.715628   58221 computation_placer.cc:177] computation placer alr

In [ ]:
from huggingface_hub import upload_file

# manually upload the video to the model card preview
upload_file(
    path_or_fileobj="./videos/ALE-SpaceInvaders-v5-step-0-to-step-5000.mp4",
    path_in_repo="replay.mp4",
    repo_id="elliemci/dqn-SpaceInvadersNoFrameskip-v4",
    repo_type="model",
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...5-step-0-to-step-5000.mp4: 100%|##########| 1.16MB / 1.16MB            

  ...5-step-0-to-step-5000.mp4: 100%|##########| 1.16MB / 1.16MB            

CommitInfo(commit_url='https://huggingface.co/elliemci/dqn-SpaceInvadersNoFrameskip-v4/commit/908ccfa14ce8e950c39bd3e79164cd6707e835c6', commit_message='Upload replay.mp4 with huggingface_hub', commit_description='', oid='908ccfa14ce8e950c39bd3e79164cd6707e835c6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/elliemci/dqn-SpaceInvadersNoFrameskip-v4', endpoint='https://huggingface.co', repo_type='model', repo_id='elliemci/dqn-SpaceInvadersNoFrameskip-v4'), pr_revision=None, pr_num=None)

In [ ]:
!pwd

/content/drive/MyDrive/ColabNotebooks/DeepRL/deep_q_learning


## Load and Evaluate a trained model

In [ ]:
# download model from the hub and save it into rl_trained folder
!python -m rl_zoo3.load_from_hub --algo dqn --env BeamRiderNoFrameskip-v4 -orga sb3 -f rl_trained/

2025-09-17 18:41:04.610581: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758134464.684012   11753 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758134464.706034   11753 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1758134464.761071   11753 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1758134464.761141   11753 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1758134464.761151   11753 computation_placer.cc:177] computation placer alr

In [ ]:
# evaluate for 5000 timesteps
!python -m rl_zoo3.enjoy --algo dqn --env BeamRiderNoFrameskip-v4 -n 5000  -f rl_trained/ --no-render

2025-09-17 18:42:52.045691: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758134572.079733   12209 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758134572.091972   12209 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1758134572.114688   12209 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1758134572.114730   12209 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1758134572.114735   12209 computation_placer.cc:177] computation placer alr

## Hyperparameter Tuning

Deep RL is far more sensitive to the choice of hyper-parameters, and variability in performance across random seeds. In addition, selecting the appropriate algorithm is of importance

In [ ]:
!pip install sb3-contrib

In [ ]:
!pip install stable-baselines3[extra]

In [ ]:
# for automatic hyperparameter tuning
!pip install optuna

In [ ]:
import gymnasium as gym
import numpy as np

from stable_baselines3 import DQN # Deep Q-Network
from sb3_contrib import QRDQN, TQC # Quantile Regression Deep Q-Network and Twin Delayed DDPG with Clipped Quantile critics

from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
#env_id = "SpaceInvadersNoFrameskip-v4"
#budget_steps = 4000

#dqn_model = DQN("CnnPolicy", env_id, seed=0, verbose=0).learn(budget_steps)
env = gym.make("CartPole-v1", render_mode="human")

model = DQN("MlpPolicy", env, verbose=1)

In [ ]:
env_id = "SpaceInvadersNoFrameskip-v4"
#eval_envs = make_vec_env(env_id, n_envs=10)
#budget_steps = 4000

In [ ]:
dqn_model = DQN("CnnPolicy", env_id, seed=0, verbose=0).learn(budget_steps)